# 10 · Customer clustering and validation

**Fieldwise Cloud · Approved unsupervised research · Great Britain**

The research asks whether one eligible canonical account per UTC month
end can be assigned to stable, interpretable financial and contractual
groups. March 2023 supplied fitting evidence. A frozen procedure was
compared on December 2023. March 2024 is a subsequent assessment date,
with earlier descriptive exposure. The separate seed-83108 population
is confirmation evidence only if a nominee clears the frozen gates.

This notebook reads checksum-verified evidence from the approved finite
experiment. It reuses the untouched-source historical reconstruction for
descriptive diagnostics. It runs no estimator fit or reserve inference.

[Scope and validation record](../../docs/05_ml/phase08_segmentation_scope_and_validation.md)
and [Notebook 09 financial evidence](09_rfm_and_customer_profiling.ipynb).

In [ ]:
from dataclasses import asdict
from decimal import Decimal
from os import getenv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.ticker import PercentFormatter
from scipy.cluster.hierarchy import dendrogram

from nowa_customer_intelligence.models.clv.reference import load_reference_history, month_ends
from nowa_customer_intelligence.models.segmentation.diagnostics import FEATURES, numeric_quality
from nowa_customer_intelligence.models.segmentation.profiles import profile_accounts
from nowa_customer_intelligence.models.segmentation.research import DEFAULT_WORKSPACE, load_report

plt.rcParams.update(
    {
        "figure.figsize": (9, 4),
        "figure.dpi": 125,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.alpha": 0.2,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
    }
)
palette = {"monthly": "#236584", "annual": "#a45324", "mixed": "#68468a", "unobserved": "#606c70"}
labels = {
    "recency_days": "Invoice recency (days)",
    "frequency": "Invoices (count)",
    "monetary_gbp": "Gross issuance (GBP)",
    "tenure_days": "Tenure (days)",
}
columns = ["recency_days", "frequency", "monetary_gbp"]


def show(fig):
    fig.tight_layout()
    plt.show()
    plt.close(fig)


ledger, inventory = load_reference_history()
observed = month_ends(ledger)[-1]


def account_frame(instant):
    rows = profile_accounts(ledger, instant)
    return pd.DataFrame.from_records(asdict(row) for row in rows).set_index(
        "customer_id", verify_integrity=True
    )


accounts = account_frame(observed)
rfm = accounts[columns].astype(float)

root = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "pyproject.toml").is_file()
)
configured = getenv("NCI_SEGMENTATION_WORKSPACE")
workspace = Path(configured) if configured else root / DEFAULT_WORKSPACE
report = load_report(workspace) if configured or workspace.exists() else None
family_colour = {"K-Means": "#236584", "Ward": "#a45324", "GMM": "#68468a"}
cluster_colour = plt.get_cmap("tab10")

## Feature dictionary and measurement rules

The approved candidate fields preserve exact Decimal GBP until
conversion at the modelling boundary. Canonical identity joins
observations for descriptive temporal checks. It remains outside the
feature spaces. Dated coverage, episode links and plan information
support diagnosis. The training-only transformation admits the
declared financial and contractual blocks.

Cumulative value, mean issue size and trailing issuance share invoice
evidence. Giving each equal weight would count the same signal several
times. Account usage needs comparable complete months before entering
a clustering block. Both candidate and excluded fields are listed.

In [ ]:
dictionary = pd.DataFrame(report["design"]["feature_dictionary"] if report else FEATURES)
with pd.option_context("display.max_colwidth", None):
    display(dictionary[["name", "calculation", "meaning", "source", "units", "dtype"]])
    display(dictionary[["name", "availability", "missing", "limitation"]])
quality = numeric_quality(accounts)
display(quality.round(3))
display(
    accounts[["invoice_feed_available", "calendar_history_complete", "usable_active_snapshots"]]
    .value_counts()
    .rename("Accounts")
    .to_frame()
)

if report:
    display(dictionary[["name", "spaces", "admission"]])

## Catalogue atoms, cadence and duplicated information

Tubes and bands in raw feature space can reflect the recurring billing
template. Exact RFM tuples quantify this concentration without using a
model. Treat each point with multiplicity as a set of observations sharing
the same measurements. Degenerate or strongly quantised features add
little separation of customer behaviour and can distort optimisation.
The coverage comparison tests whether turnover and elapsed history can
change the feature support.

In [ ]:
atoms = accounts.groupby(columns, dropna=False).size().sort_values(ascending=False)
display(atoms.head(10).rename("Accounts with exact RFM tuple").to_frame())
compare = accounts[columns + ["cadence", "calendar_history_complete"]].copy()
compare[columns] = compare[columns].astype(float)
display(
    compare.groupby(["cadence", "calendar_history_complete"])[columns]
    .agg(["size", "median"])
    .round(1)
)

fig, ax = plt.subplots(figsize=(9, 5))
for cadence, colour in palette.items():
    current = compare.loc[compare.cadence.eq(cadence)]
    if len(current):
        grouped = current.groupby(["frequency", "monetary_gbp"]).size().reset_index(name="n")
        ax.scatter(
            grouped.frequency,
            grouped.monetary_gbp,
            s=12 + 9 * np.sqrt(grouped.n),
            alpha=0.55,
            color=colour,
            label=cadence,
        )
ax.set(
    title="Billing cadence and repeated frequency–amount pairs",
    xlabel="Trailing issues (count)",
    ylabel="Trailing gross issuance (GBP)",
)
ax.legend(title="Historical cadence")
show(fig)

fig, ax = plt.subplots()
ranks = range(1, min(30, len(atoms)) + 1)
ax.bar(ranks, atoms.iloc[: len(ranks)], color="#236584")
ax.set(title="Most repeated exact RFM tuples", xlabel="Tuple rank", ylabel="Accounts")
show(fig)

observations = accounts[
    ["all_observed_gbp", "mean_issue_gbp", "monetary_gbp", "frequency", "tenure_days"]
].astype(float)
display(observations.corr(method="spearman").round(2))
display(
    Markdown(
        f"**{len(atoms):,} unique exact RFM tuples** describe {len(accounts):,} accounts. "
        f"The most common tuple occurs in {int(atoms.iloc[0]):,} accounts. "
        "An exact tie gives a distance method no evidence to distinguish those records "
        "using the three raw measures alone."
    )
)

## Distance, scaling and the assignment pipeline

**Verified observations → eligibility → feature block validation →
exact GBP conversion → standardisation and cadence encoding →
block weighting → estimator coordinates → native assignment where supported.**

For Euclidean distance \(d^2(i,j)=\sum_h(x_{ih}-x_{jh})^2\).
In the worked example below the profiles are the observed monthly and
annual medians. Changing GBP to pence makes the squared monetary
difference 10,000 times as large. Its share of the raw distance rises
from 24.74% to 99.97%. This motivates a fixed training-derived scale.

Continuous coordinates use \(z=(x-\bar x_{train})/s_{train}\).
Convert cadence to one-hot indicators using training categories.
Divide each block by \(\sqrt{d}\) with \(d\) effective coordinates.
This turns its contribution to squared distance into the average
squared difference per coordinate. Shared information and unequal
marginal variances still influence the geometry. A diagonal Gaussian
model has likelihood-based responsibilities, whose response to scaling
differs from Euclidean assignment.

Training constants are dropped and essential missing values are
rejected. A new category cannot use an arbitrary encoding. The
product-usage block failed the fixed-window availability gate and
stayed outside modelling.

In [ ]:
medians = compare.groupby("cadence")[columns].median()
squared = (medians.loc["monthly"] - medians.loc["annual"]) ** 2
pence = squared.copy()
pence["monetary_gbp"] *= 100**2
contributions = pd.DataFrame(
    {"Using GBP": squared / squared.sum(), "Using pence": pence / pence.sum()}
).T
display(medians.round(1))
display(contributions.rename(columns=labels).map(lambda value: f"{value:.2%}"))
fig, ax = plt.subplots()
contributions.rename(columns=labels).plot.bar(
    stacked=True,
    rot=0,
    color=["#236584", "#d9a56e", "#68468a"],
    ax=ax,
    title="Squared distance changes under an equivalent currency unit",
    xlabel="Raw value representation",
    ylabel="Share of squared distance",
)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3)
show(fig)

## PCA and comparison design

On a transformed fitting matrix \(Z\), the eigenvectors of
\(Z^\top Z/(n-1)\) give orthogonal directions of maximal remaining
variance. Each eigenvalue \(\lambda_h\) gives variance along that
direction. The explained share is \(\lambda_h/\sum_j\lambda_j\).
A loading describes how an original transformed dimension contributes to
a component. For a two-variable covariance with diagonal entries 1 and
off-diagonal 0.8, the equal-weight direction has eigenvalue 1.8, or 90%
of variance, and the opposite-sign direction has 0.2. This illustration
is theoretical. A projection can hide cluster separation in later
components, so the proposal compares in full transformed space and uses
component plots for display. The results below show the eigenvalues, scores and component loadings from the approved fit.

## K-Means candidate

\(J=\sum_k\sum_{i\in C_k}\|x_i-\mu_k\|^2\). Alternate nearest-centroid
assignments and within-cluster means until convergence. Multiple starts
test sensitivity to initialisation and local optima. The frozen centroids
and training transformation can support assignment of new eligible rows.
Spherical, comparably dispersed groups suit the objective. Skew, repeated
values, unequal density and arbitrary units can move centroids.

| Fitted control | Value |
|---|---|
| Groups and initialisation | 2–6 and k-means++ |
| Optimisation | 20 starts, at most 300 iterations, tolerance 0.0001 |
| Random seeds | 42 primary, plus 11, 29 and 73 for repeat investigation |

Review inertia against group count within a fixed representation. An elbow does not override support or assignment feasibility. In the illustrative points 0, 2, 10, 12, centroids 0 and 10 first give groups {0, 2} and {10, 12} and centroid updates to 1 and 11.

## Hierarchical candidate

Agglomeration starts with singleton accounts and merges pairs of groups.
Ward chooses the least increase in within-group sum of squares
\(\Delta(A,B)=\frac{n_A n_B}{n_A+n_B}\|\mu_A-\mu_B\|^2\).
A dendrogram places the accepted merge distances on its vertical axis.
Its branches aid interpretation of a cut, and repeated distances can
reflect ties. Other linkages use different criteria and give different
meanings to branch height.

| Fitted control | Value |
|---|---|
| Linkage and metric | Ward and Euclidean |
| Cuts | 2–6 groups and a descriptive dendrogram |
| Assignment | Historical descriptive comparison only unless a separate rule is approved |

The hierarchy offers no native prediction for new accounts. A nearest
representative rule would be a separate design and needs evaluation.

For illustrative singletons 0 and 2, Ward's merge cost is
\(\frac{1\cdot1}{2}(0-2)^2=2\). Merging across the larger gap from 2
to 10 costs 32. The lower within-group variance increase is preferred.


## Gaussian mixture candidate

\(p(x)=\sum_k\pi_k\mathcal{N}(x\mid\mu_k,\Sigma_k)\), with weights
\(\pi_k\ge0\) summing to one. In expectation–maximisation the
expectation step updates membership probabilities given the current
mixture. The maximisation step updates weights, means and covariances.
Small groups or duplicated values can yield unstable covariance estimates.
Compare convergence, shared support and low-confidence memberships.
Membership probabilities are conditional on the fitted mixture and should
not be interpreted as calibrated business probabilities.

| Fitted control | Value |
|---|---|
| Components and covariance | 2–6, diagonal |
| Optimisation | 10 starts, 200 iterations, tolerance 0.001 |
| Covariance regularisation | 0.000001 |
| Seeds | 42 primary, plus 11, 29 and 73 |

Compare AIC and BIC only across comparable likelihoods within this family.
A low criterion by itself gives no evidence of interpretable segments.

If two components at one point have weighted densities 0.12 and
0.08, their responsibilities are 0.60 and 0.40. A maximum probability
of 0.60 describes confidence in that fitted mixture's assignment.


## Metrics and a controlled worked example

For observation \(i\), \(a_i\) is mean distance within its cluster and
\(b_i\) the smallest mean distance to another cluster. Silhouette
\((b_i-a_i)/\max(a_i,b_i)\) measures relative separation. Review its
distribution as well as the mean. Davies–Bouldin averages the worst ratio
\((S_i+S_j)/d(\mu_i,\mu_j)\) for each cluster, using within-cluster
mean centroid distance \(S_i\). Lower is preferable. Calinski–Harabasz
is \([B/(K-1)]/[W/(n-K)]\) for between- and within-group sums of squares.
Higher is preferable on comparable data. Inertia is \(W\).
For log likelihood \(\ell\) and \(p\) free parameters,
AIC is \(2p-2\ell\) and BIC is \(p\log n-2\ell\).

Adjusted Rand compares agreement of partitions through pairs of
observations, with the expected agreement under fixed counts subtracted
in numerator and denominator. Identical partitions score 1 even if labels
have been permuted. Chance-like agreement is near 0. Negative values can
arise. Use only shared observations for comparisons and document alignment
separately for profiles. With singleton groups, degenerate labels or
coincident centroids, record undefined or unsuitable comparisons explicitly.

In [ ]:
# Two given partitions of four abstract points and a separate likelihood sketch.
from math import comb, log

pairs = np.array([[0.0, 2.0], [10.0, 12.0]])
centres = pairs.mean(axis=1)
separation = centres[1] - centres[0]
within = ((pairs - centres[:, None]) ** 2).sum()
between = pairs.shape[1] * ((centres - pairs.mean()) ** 2).sum()
total_pairs = comb(pairs.size, 2)
same_pairs_each = 2 * comb(2, 2)
expected_agreement = same_pairs_each**2 / total_pairs
metrics = {
    "Silhouette of point 0": (11 - 2) / 11,
    "Davies–Bouldin": (1 + 1) / separation,
    "Calinski–Harabasz": between / (within / (pairs.size - 2)),
    "Inertia": within,
    "AIC with 3 parameters and log likelihood −25": 2 * 3 + 2 * 25,
    "BIC with 3 parameters and 20 observations": 3 * log(20) + 2 * 25,
    "ARI after label permutation": 1.0,
    "ARI for alternating memberships": -expected_agreement / (same_pairs_each - expected_agreement),
}
display(pd.Series(metrics, name="Illustrative value").round(3).to_frame())

The fixed protocol requires at least 200 primary fitting accounts
and a minimum of 30 accounts and 5% per cluster. Repeated starts
and 20 account subsamples at 80% size test reproducibility. The
median ARI floor is 0.75, lower-tail floor is 0.60 and the
initialisation minimum is 0.75. A nominee also needs native assignment,
forward support, limited generator dominance and separation in a
shared financial representation. Convergence and high measured
separation alone cannot establish useful customer structure.

## What changes across historical observations?

The earlier and later views have overlapping accounts. A join on canonical
identity measures support changes and movement in raw metrics for common
accounts. It does not provide independent evidence or cluster migration.
Never infer migration from a change in identifiers after refitting.
The protocol requires the same frozen
transformation and assignable model, with separate counts for common,
newly observed, formerly inactive and unassignable accounts. The separate fictional reserve already has a fixed seed and Registry.

In [ ]:
landmarks = [
    instant
    for instant in month_ends(ledger)
    if (instant.year, instant.month) in {(2023, 3), (2023, 12), (2024, 3)}
]
snapshots = {instant.strftime("%b %Y"): account_frame(instant) for instant in landmarks}
audit = []
changes = {}
previous = None
for name, current in snapshots.items():
    common_ids = current.index.intersection(previous.index) if previous is not None else None
    audit.append(
        {
            "Observation": name,
            "Visible accounts": len(current),
            "Complete histories": current.calendar_history_complete.sum(),
            "Known active": current.active_episodes.gt(0).sum(),
            "Matched to prior view": len(common_ids) if common_ids is not None else None,
            "First observed since prior view": len(current.index.difference(previous.index))
            if previous is not None
            else None,
        }
    )
    if common_ids is not None:
        changes[name] = (
            current.loc[common_ids, "monetary_gbp"] - previous.loc[common_ids, "monetary_gbp"]
        ).astype(float)
    previous = current
audit = pd.DataFrame(audit).set_index("Observation")
display(audit)
display(pd.DataFrame({name: values.describe() for name, values in changes.items()}).round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(landmarks, audit["Visible accounts"], "o-", color="#236584", label="Visible accounts")
axes[0].plot(
    landmarks, audit["Complete histories"], "o-", color="#a45324", label="Complete histories"
)
axes[0].set_xticks(landmarks, labels=audit.index)
axes[0].set(title="Population and observed coverage", ylabel="Accounts", xlabel="Observation")
axes[0].legend()
axes[1].boxplot(list(changes.values()), tick_labels=list(changes))
axes[1].axhline(0, color="#606c70", linestyle="--")
axes[1].set(
    title="Matched-account value change",
    xlabel="End of comparison interval",
    ylabel="Change in trailing gross issuance (GBP)",
)
show(fig)

display(
    Markdown(
        f"March 2024 contains **{int(audit.iloc[-1]['Matched to prior view']):,}** "
        "accounts also seen in December 2023 and "
        f"**{int(audit.iloc[-1]['First observed since prior view']):,}** later entrants. "
        "Median change in gross issuance is zero among matched accounts in both "
        "comparisons. Newer accounts and outlying renewal changes still matter "
        "to a future assignment policy."
    )
)

## Executive experiment dashboard

**Temporal design** · March 2023 development fitting → December 2023
frozen candidate comparison → March 2024 subsequent assessment if
nominated. Seed 83108 is a separate fictional client and Registry at
March 2024. Its purpose is confirmation after nomination only.

Observe the change in primary eligibility rather than inferring
it from the visible roster. Fit transformations and candidate
estimators on the March 2023 cohort. Apply native candidate
assignments to later eligible accounts for comparison. Ward
contributes geometrical evidence and has no native prediction.

In [ ]:
if report:
    design = report["design"]
    experiments = report["transformations"]
    results = report["candidates"]
    date_audit = pd.DataFrame(design["feasibility"])
    display(
        date_audit[
            [
                "date",
                "visible",
                "complete_calendar",
                "eligible",
                "visible_excluded_unassignable",
                "cadence_eligible",
                "eligible_with_any_usage",
                "eligible_with_full_usage",
            ]
        ]
    )
    display(
        pd.DataFrame(
            [
                {
                    "Space": name,
                    "Effective dimensions": len(result["columns"]),
                    "Included": result["columns"],
                    "Dropped constant": result["dropped_constants"],
                }
                for name, result in experiments.items()
            ]
        )
    )
    dashboard = pd.Series(
        {
            "Candidate configurations": len(results),
            "Groups per family": design["protocol"]["groups"],
            "Converged candidates": sum(row.get("converged", False) for row in results),
            "Candidates with adequate development support": sum(
                row.get("development", {}).get("supported", False) for row in results
            ),
            "Candidates clearing every nomination gate": sum(not row["gates"] for row in results),
            "Nominee": report["nominee"] or "None",
            "Reserve status": report["reserve"]["status"].replace("_", " "),
        },
        name="Result",
    )
    display(dashboard.to_frame())
else:
    display(Markdown("Run the Phase 08 research command to render verified fitted evidence."))

## Transformation and principal component results

A standard deviation near one before block weighting would reflect
numerical standardisation. After weighting, a continuous coordinate
has standard deviation \(1/\sqrt{d}\) when its block has d
active coordinates. One-hot indicators do not have the same marginal
spread as continuous coordinates. Their covariance can be singular
because the complete indicator set sums to one.

Principal components maximise projected variance in the specified
full distance space. Component loadings help identify which measures
supply that variance. A large loading conveys influence on a
projection, without establishing causal or commercial importance.

In [ ]:
if report:
    scaled = pd.DataFrame(
        [
            {"Space": name, "Coordinate": coord, "Transformed SD": deviation}
            for name, item in experiments.items()
            for coord, deviation in zip(
                item["columns"], item["transformed_standard_deviation"], strict=True
            )
        ]
    )
    display(scaled.pivot(index="Coordinate", columns="Space", values="Transformed SD").round(3))
    display(
        pd.DataFrame(
            [
                {
                    "Space": name,
                    "Block": block["name"],
                    "Coordinates": block["coordinates"],
                    "Frozen mean": np.round(block["means"], 3),
                    "Frozen scale": np.round(block["scales"], 3),
                    "Weight": block["weight"],
                }
                for name, item in experiments.items()
                for block in item["blocks"]
            ]
        )
    )
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for name, item in experiments.items():
        ratios = np.asarray(item["pca"]["ratios"])
        components = np.arange(1, len(ratios) + 1)
        axes[0].plot(components, ratios, "o-", label=name)
        axes[1].plot(components, ratios.cumsum(), "o-", label=name)
    axes[0].set(title="Development PCA scree", xlabel="Component", ylabel="Explained share")
    axes[1].set(title="Cumulative variance", xlabel="Components retained", ylabel="Explained share")
    axes[1].legend()
    for ax in axes:
        ax.yaxis.set_major_formatter(PercentFormatter(1))
    show(fig)

    item = experiments["FS-02"]
    fig, ax = plt.subplots(figsize=(9.3, 5))
    matrix = np.asarray(item["pca"]["loadings"])[:4]
    image = ax.imshow(matrix, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set(
        title="First four FS-02 PCA loadings",
        xlabel="Development coordinate",
        ylabel="Component",
        yticks=range(len(matrix)),
        yticklabels=range(1, len(matrix) + 1),
        xticks=range(len(item["columns"])),
        xticklabels=item["columns"],
    )
    ax.tick_params(axis="x", labelrotation=43)
    ax.grid(False)
    fig.colorbar(image, ax=ax, label="Loading")
    show(fig)
    display(
        Markdown(
            "The first two components explain about **93%** in FS-01 and "
            "**75%** in FS-02. The contractual block spreads variation over more "
            "coordinates. All models use their full feature representation."
        )
    )

    fig, ax = plt.subplots(figsize=(13, 2.7))
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.axis("off")
    flow = [
        "Verified account\nand episode ledger",
        "Primary cohort\nand feature checks",
        "Training-only scale\nand block weights",
        "Full-space fit\nPCA for display",
        "Frozen validation\nand nomination gate",
    ]
    for position, label in zip(np.linspace(0.10, 0.90, len(flow)), flow, strict=True):
        ax.text(
            position,
            0.50,
            label,
            ha="center",
            va="center",
            fontsize=10,
            bbox={"boxstyle": "round,pad=0.55", "fc": "#edf3f6", "ec": "#236584"},
        )
        if position < 0.89:
            ax.annotate(
                "",
                (position + 0.13, 0.50),
                (position + 0.075, 0.50),
                arrowprops={"arrowstyle": "->", "color": "#606c70"},
            )
    ax.set_title("Phase 08 path from dated evidence to a candidate assignment")
    show(fig)

## Candidate support, separation and convergence

The model table shows all configurations. Internal distances have
different meanings in different feature spaces. Common FS-01
silhouettes enable the predeclared cross-space separation gate,
while each model's native silhouette describes its own geometry.
Empty clusters, non-convergence and unsupported groups are retained
in the audit. Validation support measures frozen assignments for
native predictive procedures.

In [ ]:
if report:
    candidate_table = pd.DataFrame(
        [
            {
                "Space": row["space"],
                "Family": row["family"],
                "Groups": row["groups"],
                "Accounts": row["fitting_accounts"],
                "Converged": row.get("converged", False),
                "Sizes": row.get("development", {}).get("counts"),
                "Silhouette": row.get("development", {}).get("silhouette"),
                "Common fit silhouette": row.get("common_silhouette"),
                "DB": row.get("development", {}).get("davies_bouldin"),
                "CH": row.get("development", {}).get("calinski_harabasz"),
                "Common validation silhouette": row.get("validation", {}).get("common_silhouette"),
                "Validation sizes": row.get("validation", {}).get("counts"),
                "Gates": ", ".join(row["gates"]),
            }
            for row in results
        ]
    )
    with pd.option_context("display.max_rows", 48, "display.max_colwidth", None):
        display(candidate_table.round(3))
    failures = pd.Series([gate for row in results for gate in row["gates"]]).value_counts()
    display(failures.rename("Configurations").to_frame())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.7), sharex=True)
    for name, rows in candidate_table[candidate_table.Space.eq("FS-01")].groupby("Family"):
        axes[0].plot(rows.Groups, rows.Silhouette, "o-", color=family_colour[name], label=name)
        sizes = rows.Sizes.map(min)
        axes[1].plot(rows.Groups, sizes, "o-", color=family_colour[name], label=name)
    axes[0].set(
        title="FS-01 development silhouette by family", ylabel="Silhouette", xlabel="Groups"
    )
    axes[1].axhline(30, ls="--", color="#606c70", label="Support floor")
    axes[1].set(title="FS-01 smallest fitted cluster", ylabel="Accounts", xlabel="Groups")
    axes[1].legend()
    show(fig)

    visible_validation = candidate_table.dropna(subset=["Common validation silhouette"])
    fig, ax = plt.subplots(figsize=(8.3, 5))
    for family, rows in visible_validation.groupby("Family"):
        ax.scatter(
            rows["Common fit silhouette"],
            rows["Common validation silhouette"],
            color=family_colour[family],
            s=25 + rows.Groups * 6,
            alpha=0.6,
            label=family,
        )
    ax.axhline(0.20, color="#606c70", ls="--")
    ax.axvline(0.20, color="#606c70", ls="--")
    ax.set(
        title="Candidate separation in the common financial space",
        xlabel="March 2023 silhouette",
        ylabel="December 2023 frozen silhouette",
    )
    ax.legend(title="Native assignment family")
    show(fig)

## K-Means, Ward and mixture diagnostics

K-Means inertia decreases as added centroids split the observed points.
The 30-account floor rejects some high-silhouette cuts in the same
representation. Ward merge heights describe successive within-cluster
variance increases. The dendrogram compresses lower branches into
leaves labelled by their contained account counts.

AIC and BIC are shown only within the same Gaussian feature
representation. Compare them alongside support and conditional
membership confidence. The fictional monetary features contain
repeated catalogue atoms and do not follow a smooth Gaussian density.

In [ ]:
if report:
    lookup = {row["id"]: row for row in results}
    family_details = pd.DataFrame(
        [
            {
                "Space": row["space"],
                "Family": row["family"],
                "Groups": row["groups"],
                **{
                    key: row["model_details"].get(key)
                    for key in ("inertia", "iterations", "aic", "bic", "weights")
                },
            }
            for row in results
            if "model_details" in row
        ]
    )
    display(family_details[family_details.Family.ne("Ward")].round(2))

    fig, ax = plt.subplots()
    for name, rows in family_details[family_details.Family.eq("K-Means")].groupby("Space"):
        ax.plot(rows.Groups, rows.inertia, "o-", label=name)
    ax.set(
        title="K-Means inertia within each representation",
        xlabel="Cluster count",
        ylabel="Weighted squared distance",
    )
    ax.legend()
    show(fig)

    ward = experiments["FS-01"]["representative"]["Ward"]
    hierarchy = np.asarray(ward["hierarchy"])
    display(pd.Series(hierarchy[-8:, 2], name="Largest merge distances"))
    fig, ax = plt.subplots(figsize=(11, 4.6))
    dendrogram(
        hierarchy,
        ax=ax,
        truncate_mode="lastp",
        p=14,
        above_threshold_color="#606c70",
        leaf_font_size=9,
    )
    ax.set(
        title="Ward hierarchy for FS-01 · truncated to last 14 branches",
        xlabel="Branch size or singleton row",
        ylabel="Euclidean Ward merge height",
    )
    show(fig)

In [ ]:
if report:
    gmm_rows = family_details[family_details.Family.eq("GMM")]
    fig, axes = plt.subplots(1, 3, figsize=(13.2, 4))
    for (space, rows), ax in zip(gmm_rows.groupby("Space"), axes, strict=True):
        ax.plot(rows.Groups, rows.aic, "o-", label="AIC", color="#236584")
        ax.plot(rows.Groups, rows.bic, "s--", label="BIC", color="#a45324")
        ax.set(title=space, xlabel="Components", ylabel="Penalised likelihood criterion")
    axes[0].legend()
    fig.suptitle("Gaussian mixture criteria · comparisons within each panel")
    show(fig)
    mixture_case = experiments["FS-01"]["representative"]["GMM"]
    mixture = lookup[mixture_case["id"]]
    mixture_details = mixture["model_details"]
    display(
        pd.DataFrame(
            {
                "Weight": mixture_details["weights"],
                "Accounts": mixture["development"]["counts"],
                "Covariance range": [
                    (min(axis), max(axis)) for axis in mixture_details["covariances"]
                ],
            }
        ).rename_axis("Component")
    )
    fig, ax = plt.subplots()
    confidence = pd.Series(mixture_details["membership_confidence"])
    ax.hist(confidence, bins=np.linspace(0, 1, 21), color="#68468a")
    ax.set(
        title=f"Conditional membership confidence · {len(confidence)} development accounts",
        xlabel="Maximum fitted membership probability",
        ylabel="Accounts",
        xlim=(0, 1),
    )
    show(fig)

## Development partitions and customer profiles

The geometric comparison uses a supported descriptive fit in each
family. These displayed candidates have no nominated business labels.
Projection and optimisation use different dimensionalities. The
panels communicate visible geometry, while silhouettes quantify the
full transformed space. Show negative silhouettes separately from
small clusters: an account can be less similar to its assigned group
even when every group exceeds the support floor.

Original-unit summaries help interpret any apparent separation.
Group means in transformed coordinates identify the pattern the
estimator actually uses. Both views account for the same set of
development records.

In [ ]:
if report:
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.1))
    for example, ax in zip(experiments["FS-01"]["representative"].values(), axes, strict=True):
        points, membership = np.asarray(example["projected"]), np.asarray(example["labels"])
        for label in np.unique(membership):
            mask = membership == label
            ax.scatter(
                points[mask, 0],
                points[mask, 1],
                s=14,
                alpha=0.58,
                color=cluster_colour(label),
                label=f"G{label} · {mask.sum()}",
            )
        ax.set(title=example["id"].split(" / ", 1)[1], xlabel="PC1 score", ylabel="PC2 score")
        ax.legend(fontsize=8)
    fig.suptitle("FS-01 development PCA overlays · frozen display projection")
    show(fig)

    reference_fit = lookup["FS-01 / K-Means / 2"]
    profile = reference_fit["profiles"]
    group_rows = pd.DataFrame(profile["groups"]).T
    display(group_rows[["accounts", "share", "issued_gbp", "cadence"]])
    assert group_rows.accounts.sum() == profile["accounts"]
    assert sum(map(Decimal, group_rows.issued_gbp), Decimal(0)) == Decimal(profile["issued_gbp"])

    silhouette = reference_fit["development"]["silhouette_by_group"]
    fig, ax = plt.subplots()
    bottom = 0
    for group, scores in silhouette.items():
        ordered = np.sort(scores)
        indices = np.arange(bottom, bottom + len(scores))
        ax.fill_betweenx(
            indices,
            0,
            ordered,
            color=cluster_colour(int(group)),
            alpha=0.8,
            label=f"G{group} · {len(scores)}",
        )
        bottom += len(scores) + 5
    ax.axvline(
        reference_fit["development"]["silhouette"], ls="--", color="#606c70", label="Overall mean"
    )
    ax.set(
        title="FS-01 K-Means with two groups · sorted silhouettes",
        xlabel="Individual silhouette",
        ylabel="Ordered accounts and group gaps",
    )
    ax.legend()
    show(fig)
    display(
        pd.Series(
            {
                f"G{group}": sum(value < 0 for value in scores)
                for group, scores in silhouette.items()
            },
            name="Negative silhouettes",
        )
    )

In [ ]:
if report:
    original_fields = [
        "recency_days",
        "frequency",
        "monetary_gbp",
        "tenure_days",
        "contractual_gbp_per_month",
        "reactivated_episodes",
        "observed_usage_periods",
        "mean_active_users",
        "mean_sessions",
        "mean_events",
    ]
    original_profile = pd.DataFrame(
        [
            {"Cluster": f"G{group}", "Measure": field, **record["statistics"][field]}
            for group, record in profile["groups"].items()
            for field in original_fields
        ]
    )
    display(original_profile.round(2))
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.4))
    for measure, ax in zip(["recency_days", "frequency", "monetary_gbp"], axes, strict=True):
        stats = [
            {
                "label": row["Cluster"],
                "med": row["50%"],
                "q1": row["25%"],
                "q3": row["75%"],
                "whislo": row["min"],
                "whishi": row["max"],
                "fliers": [],
            }
            for _, row in original_profile[original_profile.Measure.eq(measure)].iterrows()
        ]
        ax.bxp(stats, showfliers=False)
        ax.set(
            xlabel="Development group", ylabel=labels[measure], title="Observed range and quartiles"
        )
    show(fig)

    centre = np.stack(group_rows.transformed_mean)
    fig, ax = plt.subplots(figsize=(8, 3.7))
    magnitude = np.abs(centre).max()
    image = ax.imshow(centre, cmap="RdBu_r", vmin=-magnitude, vmax=magnitude, aspect="auto")
    ax.set(
        title="FS-01 mean profiles in weighted coordinates",
        xlabel="Feature",
        ylabel="Development group",
        xticks=range(len(experiments["FS-01"]["columns"])),
        xticklabels=experiments["FS-01"]["columns"],
        yticks=range(len(centre)),
        yticklabels=[f"G{label}" for label in group_rows.index],
    )
    ax.grid(False)
    fig.colorbar(image, ax=ax, label="Mean weighted standard units")
    show(fig)

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.9))
    activity = {
        "mean_active_users": "Active users",
        "mean_sessions": "Sessions",
        "mean_events": "Events",
    }
    for (measure, title), ax in zip(activity.items(), axes, strict=True):
        observed_usage = original_profile.loc[original_profile.Measure.eq(measure)]
        midpoint = observed_usage["50%"].to_numpy()
        tails = np.vstack([midpoint - observed_usage["25%"], observed_usage["75%"] - midpoint])
        ax.errorbar(
            observed_usage.Cluster, midpoint, yerr=tails, fmt="o", color="#68468a", capsize=5
        )
        ax.set(
            title=title, xlabel="Financial development group", ylabel="Count/month · median and IQR"
        )
    fig.suptitle("Descriptive usage in available months · excluded from clustering")
    show(fig)

## Generator influence and indirect cadence

ARI between feature spaces compares the same development accounts
and is invariant to arbitrary label swaps. NMI and cadence homogeneity
describe association with known generator characteristics. A high
cadence result for FS-01 shows that recency, count and issued amounts
already encode it even when no explicit cadence field is present.

Removing direct cadence may move the dominant pattern elsewhere,
including synthetic reactivation. A low cadence association alone
cannot establish discovery of useful behavioural structure.

In [ ]:
if report:
    reference_rows = [
        "FS-01 / K-Means / 2",
        "FS-02 / K-Means / 2",
        "FS-02 without cadence / K-Means / 2",
    ]
    association_table = pd.DataFrame(
        {name: lookup[name]["associations"]["nmi"] for name in reference_rows}
    ).T
    display(association_table.round(3))
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
    counts = pd.DataFrame(reference_fit["associations"]["cadence_counts"]).T
    shares = counts.div(counts.sum(axis=0), axis=1).T
    shares.plot.bar(
        stacked=True,
        rot=0,
        color=palette,
        ax=axes[0],
        title="FS-01 K-Means cadence composition",
        ylabel="Group share",
    )
    axes[0].set_xlabel("Development group")
    axes[0].yaxis.set_major_formatter(PercentFormatter(1))
    image = axes[1].imshow(association_table, cmap="Purples", vmin=0, vmax=1, aspect="auto")
    axes[1].set(
        title="NMI against fixed generator references",
        xticks=range(len(association_table.columns)),
        xticklabels=association_table.columns,
        yticks=range(3),
        yticklabels=["FS-01", "FS-02", "FS-02 without cadence"],
    )
    axes[1].tick_params(axis="x", labelrotation=40)
    axes[1].grid(False)
    fig.colorbar(image, ax=axes[1], label="Normalised mutual information")
    show(fig)

    sensitivity = pd.DataFrame(report["sensitivity"])
    sensitivity["Family"] = sensitivity.left.str.split(" / ").str[-2]
    display(sensitivity.groupby("Family").ari.agg(["min", "median", "max"]).round(3))
    display(
        sensitivity.loc[
            sensitivity.left.isin(reference_rows) & sensitivity.right.isin(reference_rows)
        ].round(3)
    )

## Initialisation and account resampling stability

Each repeated seed uses the same ordered account matrix. The
deterministic Ward candidate has no initialisation experiment.
Subsampling uses the same predetermined 80% account subsets for
the conditional and complete workflow modes. The adjusted comparing
labels use identical indices for each subset. Complete workflow rebuilds
the scaler and category support from that subset. Each experiment
retains convergence, group sizes, shared support and failed trials.

Tight ARI distributions show reproducibility under these alterations.
They can still reproduce a deterministic billing pattern.

In [ ]:
if report:
    repeated = [row for row in results if "stability" in row]
    initial = pd.DataFrame(
        [
            {"Candidate": row["id"]} | trial
            for row in repeated
            for trial in row["stability"]["initialisations"]
        ]
    )
    display(initial)
    modes = ("conditional", "complete_workflow")
    stability_rows = []
    distributions = []
    for row in repeated:
        replicates = row["stability"]["resamples"]
        for mode in modes:
            values = [trial[mode]["ari"] for trial in replicates if trial[mode]["ari"] is not None]
            groups = [min(trial[mode]["sizes"]) for trial in replicates if "sizes" in trial[mode]]
            distributions.append({"candidate": row["id"], "mode": mode, "values": values})
            stability_rows.append(
                {
                    "Candidate": row["id"],
                    "Mode": mode,
                    "Median ARI": np.median(values),
                    "10th percentile": np.quantile(values, 0.1),
                    "IQR": np.subtract(*np.quantile(values, [0.75, 0.25])),
                    "Completed": len(values),
                    "Failures": len(replicates) - len(values),
                    "Shared accounts": replicates[0]["shared_accounts"],
                    "Smallest trial cluster": min(groups),
                }
            )
    display(pd.DataFrame(stability_rows).round(3))
    fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)
    abbreviations = [
        row["id"].replace("FS-02 without cadence", "FS-02 no cadence") for row in repeated
    ]
    for mode, ax in zip(modes, axes, strict=True):
        values = [row["values"] for row in distributions if row["mode"] == mode]
        ax.boxplot(values, orientation="horizontal", tick_labels=abbreviations)
        ax.axvline(0.75, color="#a45324", ls="--", label="Median gate")
        ax.axvline(0.60, color="#606c70", ls=":", label="Lower-tail gate")
        ax.set(title=mode.replace("_", " ").capitalize(), xlabel="Adjusted Rand index")
        ax.legend(loc="lower left")
    fig.suptitle("Reproducibility of supported development fits · 20 account subsets")
    show(fig)

    size_matrix = pd.DataFrame(stability_rows).pivot(
        index="Candidate", columns="Mode", values="Smallest trial cluster"
    )
    fig, ax = plt.subplots(figsize=(10.5, 7))
    size_matrix.plot.barh(
        ax=ax,
        color=["#236584", "#a45324"],
        title="Smallest observed group across resampling trials",
        xlabel="Accounts in the smallest trial cluster",
        ylabel="Development candidate",
    )
    ax.axvline(30, color="#606c70", ls="--", label="Full-sample minimum")
    ax.legend()
    show(fig)

## Nomination, frozen transfer and reserve protection

The engineering decision is negative. It preserves the differences
between stable but generator-dominated fits and descriptive fits
with weak financial separation. The development FS-01 two-cluster
K-Means partition has cluster support, strong separation and high
resampling stability, but reproduces billing cadence. The FS-02
no-cadence four-group K-Means alternative has useful development
support and stability. Its common financial-space separation fails
the predetermined gate at development and validation.

December 2023 native assignments were used for candidate comparison.
Without a nominee there is no March 2024 model application or
independent reserve scoring. The separately frozen RFM references
can be compared across the historical roster in their original role
as financial descriptors. Transition counts use shared assignments,
and roster entries and exits keep separate denominators.

In [ ]:
if report:
    named_cases = ("FS-01 / K-Means / 2", "FS-02 without cadence / K-Means / 4")
    display(
        candidate_table.loc[
            candidate_table.apply(
                lambda row: f"{row.Space} / {row.Family} / {row.Groups}" in named_cases, axis=1
            )
        ]
    )
    frozen_evidence = report["reserve"]
    display(pd.Series(frozen_evidence["feasibility"], name="Seed 83108 count-only evidence"))
    display(
        pd.Series(
            {
                "Nominee": report["nominee"] or "None",
                "Subsequent model transfer": "Available" if report["temporal"] else "Unavailable",
                "Reserve model evaluation": frozen_evidence["status"].replace("_", " "),
            },
            name="Frozen procedure disposition",
        )
    )
    history = [
        {
            "Date": snapshot["date"],
            "Reference": method,
            **{k: v for k, v in result.items() if k != "transition"},
        }
        for snapshot in report["rfm"]
        if snapshot["frozen_reference_migration"]
        for method, result in snapshot["frozen_reference_migration"].items()
    ]
    display(pd.DataFrame(history))
    final_transition = report["rfm"][-1]["frozen_reference_migration"]["traditional"]
    matrix = (
        pd.DataFrame.from_dict(final_transition["transition"], orient="index").fillna(0).astype(int)
    )
    fig, ax = plt.subplots(figsize=(8.3, 5.6))
    image = ax.imshow(matrix, cmap="Blues", aspect="auto")
    ax.set(
        title=f"Traditional RFM migration · {final_transition['shared_assigned']} shared accounts",
        xlabel="March 2024 frozen score",
        ylabel="December 2023 frozen score",
        xticks=range(len(matrix.columns)),
        xticklabels=matrix.columns,
        yticks=range(len(matrix)),
        yticklabels=matrix.index,
    )
    ax.tick_params(axis="x", labelrotation=45)
    ax.grid(False)
    fig.colorbar(image, ax=ax, label="Accounts")
    show(fig)

## Business reading and reproducibility

The descriptive partitions can help Revenue Operations inspect
exposure and renewal cadence. Their synthetic regularity and strong
reactivation and price effects give no evidence of retention uplift,
independent customer types or a validated decision rule. Customer
Success could review raw historical facts with context, including
coverage for the behavioural data.

The signed-off experiment definition was frozen before development
fits. The compatibility of the fictional source, prior limited
modelling evidence, and the separate reserve are recorded in the
report manifest. Future commercial research would require validated
production source availability, independent customer evidence and
approved rules for usage and assignment.

Reopen this view in a fresh Python 3.12 kernel with a local
checksum-verified report. The descriptive references also run
offline. No model object or independent source data is loaded by this
notebook. [Reproduction instructions](../README.md#phase-08-customer-segmentation).